# Topico 1: Entropia, Informacao Mutua e o Teste de Independencia

Cenario: teste rapido de dengue. D = estado real, T = resultado do teste. Enunciado: `propostas/1_Simulacoes.pdf`, secao 1.

## Preparacao

In [ ]:
import sys, pathlib

REPO = pathlib.Path.cwd().parents[2]          # .../<topico>/ -> .../<proposta>/ -> raiz
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from utils.info_theory import (
    entropy, binary_entropy, joint_entropy, conditional_entropy,
    cross_entropy, kl_divergence, mutual_information,
    conditional_mutual_information, mi_gtest_statistic, counts_to_joint,
)
from utils.simulation import (
    rng, sample_pmf, sample_joint, empirical_pmf, bsc, markov_chain,
)
from utils.estimation import bootstrap_ci
from utils.plotting import (
    setup_estilo, salvar_fig, plot_barras, tabela_conjunta, tabela_comparacao,
    plot_estimativa_vs_n, plot_curvas,
)
import src.model as model                     # helpers especificos deste topico

setup_estilo()
GEN = rng(20260902)                           # seed fixo -> resultados reproduziveis
DADOS = pathlib.Path("dados")
FIGURAS = pathlib.Path("figuras")

## Passo 1: Modelo verdadeiro

Definir P(D=1), P(T=1|D=0) e P(T=0|D=1) (`model.true_joint`). Calcular os valores TEORICOS de H(D), H(T), H(D,T), H(D|T) e I(D;T).

In [ ]:
# Passo 1
PREVALENCIA = 0.08   # P(D = 1)
FPR = 0.03           # P(T = 1 | D = 0), falso positivo
FNR = 0.12           # P(T = 0 | D = 1), falso negativo

JOINT_TEORICO = model.true_joint(PREVALENCIA, FPR, FNR)
display(tabela_conjunta(JOINT_TEORICO, rotulos_x=["D=0", "D=1"], rotulos_y=["T=0", "T=1"]))

P_D_TEORICO = JOINT_TEORICO.sum(axis=1)
P_T_TEORICO = JOINT_TEORICO.sum(axis=0)

TEORICO = {
    "H(D)": entropy(P_D_TEORICO),
    "H(T)": entropy(P_T_TEORICO),
    "H(D,T)": joint_entropy(JOINT_TEORICO),
    "H(D|T)": conditional_entropy(JOINT_TEORICO, given="y"),  # H(X | Y) = H(D | T)
    "I(D;T)": mutual_information(JOINT_TEORICO),
}
for nome, valor in TEORICO.items():
    print(f"{nome} = {valor:.4f} bits")

plot_barras(TEORICO, ylabel="bits")

## Passo 2: Amostra de N pares (D, T)

Gerar N pacientes com `model.sample_pairs` (sorteia D pela marginal, T condicional a D).

In [ ]:
# Passo 2
N = 2000  # tamanho da amostra "N fixo" usada na comparacao do Passo 3

D_AMOSTRA, T_AMOSTRA = model.sample_pairs(JOINT_TEORICO, N, GEN)
print(f"N = {N} pares (D, T) gerados")
print(f"prevalencia empirica de D: {D_AMOSTRA.mean():.4f}  (teorica: {PREVALENCIA})")
print(f"positividade empirica de T: {T_AMOSTRA.mean():.4f}  (teorica: {P_T_TEORICO[1]:.4f})")

## Passo 3: Estimativas na amostra

Montar a tabela conjunta empirica (`counts_to_joint`) e reestimar H(D), H(T), H(D,T), H(D|T), I(D;T). Comparar com o Passo 1.

In [ ]:
# Passo 3
JOINT_AMOSTRA = counts_to_joint(D_AMOSTRA, T_AMOSTRA, size_x=2, size_y=2)
display(tabela_conjunta(JOINT_AMOSTRA, rotulos_x=["D=0", "D=1"], rotulos_y=["T=0", "T=1"], normalizar=True))

P_D_AMOSTRA = JOINT_AMOSTRA.sum(axis=1)
P_T_AMOSTRA = JOINT_AMOSTRA.sum(axis=0)

ESTIMADO = {
    "H(D)": entropy(P_D_AMOSTRA),
    "H(T)": entropy(P_T_AMOSTRA),
    "H(D,T)": joint_entropy(JOINT_AMOSTRA),
    "H(D|T)": conditional_entropy(JOINT_AMOSTRA, given="y"),
    "I(D;T)": mutual_information(JOINT_AMOSTRA),
}

display(tabela_comparacao(list(TEORICO), list(ESTIMADO.values()), list(TEORICO.values())))

## Passo 4: Erro de I(D;T) vs N

Repetir a estimativa de I para N = 20, 50, 100, 500, 2000, 10000 e observar o erro vs o valor teorico.

In [ ]:
# Passo 4
def _i_dt(d, t):
    return mutual_information(counts_to_joint(d, t, size_x=2, size_y=2))


NS = [20, 50, 100, 500, 2000, 10000]
I_ESTIMADO, I_IC_BAIXO, I_IC_ALTO = [], [], []
for n in NS:
    d_n, t_n = model.sample_pairs(JOINT_TEORICO, n, GEN)
    est, baixo, alto = bootstrap_ci((d_n, t_n), _i_dt, generator=GEN, paired=True)
    I_ESTIMADO.append(est)
    I_IC_BAIXO.append(baixo)
    I_IC_ALTO.append(alto)

display(tabela_comparacao(
    [f"N={n}" for n in NS], I_ESTIMADO, [TEORICO["I(D;T)"]] * len(NS), I_IC_BAIXO, I_IC_ALTO,
))

plot_estimativa_vs_n(
    NS, I_ESTIMADO, I_IC_BAIXO, I_IC_ALTO,
    teorico=TEORICO["I(D;T)"], label="I(D;T) estimado", ylabel="bits",
)
salvar_fig("t1_passo4_erro_i_vs_n")

## Passo 5: Entropia cruzada e divergencia

Escolher q(D) != p(D), calcular H(p, q) e D(p||q). Conferir que o excesso de custo = D(p||q).

In [ ]:
# Passo 5
Q_D = np.array([0.85, 0.15])  # q(D) adivinhado, diferente de p(D) = [1 - PREVALENCIA, PREVALENCIA]

H_PQ = cross_entropy(P_D_TEORICO, Q_D)
KL_PQ = kl_divergence(P_D_TEORICO, Q_D)
EXCESSO = H_PQ - TEORICO["H(D)"]

print(f"p(D)          = {P_D_TEORICO}")
print(f"q(D)          = {Q_D}")
print(f"H(p, q)       = {H_PQ:.4f} bits")
print(f"H(p)          = {TEORICO['H(D)']:.4f} bits")
print(f"H(p,q) - H(p) = {EXCESSO:.4f} bits")
print(f"D(p||q)       = {KL_PQ:.4f} bits")
assert np.isclose(EXCESSO, KL_PQ), "H(p,q) - H(p) deveria bater com D(p||q)"

plot_barras({"H(p)": TEORICO["H(D)"], "H(p,q)": H_PQ, "D(p||q)": KL_PQ}, ylabel="bits")

## Passo 6: Dimensao temporal (Markov)

Simular o historico T_1..T_n de um paciente com uma cadeia de Markov de 2 estados (`markov_chain`).

In [ ]:
# Passo 6
P_D_MARKOV = np.array([
    [0.95, 0.05],  # D=0 -> D=0 / D=1 no dia seguinte
    [0.20, 0.80],  # D=1 -> D=0 / D=1 no dia seguinte (infeccao persiste)
])  # cadeia de Markov de 2 estados para D, bem mais persistente que D i.i.d.
N_DIAS = 14

D_SERIE = markov_chain(P_D_MARKOV, N_DIAS, GEN, x0=1)  # paciente comeca infectado
T_SERIE = model.sample_given_d(JOINT_TEORICO, D_SERIE, GEN)

print("D_1..D_n:", D_SERIE)
print("T_1..T_n:", T_SERIE)

SERIE_DF = pd.DataFrame({
    "dia": np.tile(np.arange(1, N_DIAS + 1), 2),
    "valor": np.concatenate([D_SERIE, T_SERIE]),
    "variavel": ["D"] * N_DIAS + ["T"] * N_DIAS,
})
plot_curvas(SERIE_DF, x="dia", y="valor", hue="variavel", xlabel="dia", ylabel="estado (0/1)")
salvar_fig("t1_passo6_serie_d_t")


## Passo 7: Regra da cadeia

Estimar H(T_1..T_n); comparar com a soma das entropias condicionais e com n*H(T_1).

In [ ]:
# Passo 7
N_CADEIA = 8          # dias por paciente nesta estimativa (2^8 = 256 sequencias possiveis)
M_PACIENTES = 20000   # repeticoes independentes de markov_chain + sample_given_d

T_SEQS = np.empty((M_PACIENTES, N_CADEIA), dtype=int)
for m in range(M_PACIENTES):
    d_m = markov_chain(P_D_MARKOV, N_CADEIA, GEN)
    T_SEQS[m] = model.sample_given_d(JOINT_TEORICO, d_m, GEN)

# H(T_1,...,T_n) estimado por contagem direta sobre o alfabeto 2^n (cada sequencia -> um inteiro)
POTENCIAS_DE_2 = 2 ** np.arange(N_CADEIA - 1, -1, -1)
CODIGOS_SEQ = T_SEQS @ POTENCIAS_DE_2
P_SEQ_ESTIMADO = empirical_pmf(CODIGOS_SEQ, size=2**N_CADEIA)
H_SEQ_ESTIMADO = entropy(P_SEQ_ESTIMADO)

# soma das H(T_i | T_{i-1}) pela regra da cadeia (T_1 entra sem condicionante)
H_T1 = entropy(empirical_pmf(T_SEQS[:, 0], size=2))
SOMA_COND = H_T1
for i in range(1, N_CADEIA):
    joint_i = counts_to_joint(T_SEQS[:, i - 1], T_SEQS[:, i], size_x=2, size_y=2)
    SOMA_COND += conditional_entropy(joint_i, given="x")  # H(T_i | T_{i-1})

N_VEZES_H_T1 = N_CADEIA * H_T1

print(f"H(T_1,...,T_{N_CADEIA}) estimado (contagem 2^n) = {H_SEQ_ESTIMADO:.4f} bits")
print(f"soma H(T_i | T_(i-1))   (regra da cadeia)      = {SOMA_COND:.4f} bits")
print(f"{N_CADEIA} x H(T_1)     (T_i i.i.d. hipotetico) = {N_VEZES_H_T1:.4f} bits")

plot_barras(
    {
        "H(T_1..T_n) contagem": H_SEQ_ESTIMADO,
        "soma H(T_i|T_i-1)": SOMA_COND,
        "n*H(T_1)": N_VEZES_H_T1,
    },
    ylabel="bits",
)
salvar_fig("t1_passo7_regra_cadeia")


## Passo 8: Economia de bits

Quantificar em bits a economia de transmitir o historico de forma condicional em vez de dia a dia isolado.

In [ ]:
# Passo 8
# Comparacao entre os dois ultimos valores do Passo 7: transmitir cada dia
# isolado (n*H(T_1)) vs. transmitir condicional ao dia anterior (soma H(T_i|T_i-1)).
ECONOMIA_BITS = N_VEZES_H_T1 - SOMA_COND
ECONOMIA_PCT = 100.0 * ECONOMIA_BITS / N_VEZES_H_T1

print(f"custo dia a dia isolado, {N_CADEIA} x H(T_1)   = {N_VEZES_H_T1:.4f} bits")
print(f"custo condicional, soma H(T_i|T_i-1)   = {SOMA_COND:.4f} bits")
print(f"economia                               = {ECONOMIA_BITS:.4f} bits ({ECONOMIA_PCT:.1f}% do custo isolado)")

plot_barras(
    {"n*H(T_1) (isolado)": N_VEZES_H_T1, "soma H(T_i|T_i-1) (condicional)": SOMA_COND, "economia": ECONOMIA_BITS},
    ylabel="bits",
)
salvar_fig("t1_passo8_economia_bits")


## Passo 9: Teste de independencia (teste-G)

Aplicar `mi_gtest_statistic` a uma amostra; reportar G, g.l. e p-valor (`scipy.stats.chi2.sf`) para D vs T e para T_i vs T_{i-1}.

In [ ]:
# Passo 9


## Passo 10: Taxa de erro do teste

Repetir o experimento em 3 cenarios (dependencia forte, fraca, independencia real) e medir com que frequencia o teste acerta a decisao; taxa de rejeicao sob H0 verdadeira.

In [ ]:
# Passo 10


## Relatorio

Consolidar resultados, tabelas e figuras em `relatorio.md` (figuras salvas em `figuras/`).